# 🏠 EDA — Referencia del Precio del Alquiler en España
**Equipo:** Enrique Algarra · Lucía Vetrano · Nil Coronado  
**Bootcamp Data Science Online — The Bridge**  
**Fecha de entrega:** 22 de mayo de 2026

---

## 🎯 Pregunta de investigación

> *¿Qué variables explican mejor el precio del alquiler en España y cómo varían según la ubicación y el tipo de vivienda?*

---

## 📋 Índice

1. [Importación de librerías y carga de datos](#1-importación-de-librerías-y-carga-de-datos)
2. [Limpieza y preparación del dataset](#2-limpieza-y-preparación-del-dataset)
3. [Análisis univariante](#3-análisis-univariante)
4. [Análisis bivariante — Hipótesis](#4-análisis-bivariante--hipótesis)
   - H1: Las grandes provincias tienen precios superiores
   - H2: Existe desigualdad territorial clara
   - H3: El precio aumenta progresivamente con el tiempo
   - H4: El tipo de vivienda influye en el precio
5. [Análisis multivariante](#5-análisis-multivariante)
6. [Validación de hipótesis](#6-validación-de-hipótesis)
7. [Conclusiones finales](#7-conclusiones-finales)


## 1. Importación de librerías y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual global
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150

print("Librerías cargadas correctamente ✓")

In [ ]:
# Carga del dataset original
df_raw = pd.read_csv('src/data/raw/VDP001_01.csv', sep=';')

print(f"Dimensiones del dataset original: {df_raw.shape}")
df_raw.head()

## 2. Limpieza y preparación del dataset

El proceso de limpieza fue desarrollado por **Nil Coronado** e incluyó las siguientes etapas:
- Normalización de nombres de columnas
- Conversión de tipos de datos
- Detección y gestión de valores nulos
- Eliminación de duplicados
- Exportación del dataset limpio


In [ ]:
# Normalizar nombres de columnas
df = df_raw.copy()
df.columns = [c.strip().upper().replace(' ', '_') for c in df.columns]
print("Columnas:", list(df.columns))

In [ ]:
# Conversión de tipos de datos
df['AÑO'] = pd.to_numeric(df['AÑO'], errors='coerce')
df['VALOR'] = df['VALOR'].astype(str).str.replace(',', '.')
df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')

print("Tipos de datos tras conversión:")
print(df.dtypes)

In [ ]:
# Comprobación de valores nulos
nulos = df.isnull().sum()
print("Valores nulos por columna:")
print(nulos)
print(f"\nTotal filas con nulos: {df.isnull().any(axis=1).sum()}")

In [ ]:
# Comprobación y eliminación de duplicados
duplicados = df.duplicated().sum()
print(f"Duplicados encontrados: {duplicados}")
df = df.drop_duplicates()
print(f"Dimensiones tras limpieza: {df.shape}")

In [ ]:
# Estadísticas descriptivas generales
df.describe()

In [ ]:
# Guardar dataset limpio
df.to_csv('src/data/processed/VDP001_01_clean.csv', index=False)
print("Dataset limpio guardado en src/data/processed/ ✓")

## 3. Análisis univariante

Antes de responder las hipótesis, exploramos la distribución general del índice de precio mediano para entender la estructura del mercado.

> **Nota:** La columna `VALOR` representa **índices de precio relativos del INE**, no precios en euros. Permiten comparar evolución temporal y diferencias entre zonas, pero no reflejan el precio absoluto del alquiler.


In [ ]:
# Filtrado: solo precio mediano por municipio
# ELEMENTO = 'PRECIO' y TIPO_MEDIDA = 'MEDIANA' es la métrica más representativa
# La mediana es preferible a la media porque no se ve distorsionada por valores extremos
df_precio = df[(df['ELEMENTO'] == 'PRECIO') & (df['TIPO_MEDIDA'] == 'MEDIANA')]
print(f"Registros de precio mediano: {df_precio.shape[0]:,}")
print(f"Años disponibles: {sorted(df_precio['AÑO'].unique())}")
print(f"Provincias: {df_precio['PROVINCIA'].nunique()}")

In [ ]:
# Estadísticas descriptivas del índice de precio mediano
df_precio['VALOR'].describe().round(1)

In [ ]:
# Distribución general del índice de precio mediano
plt.figure(figsize=(10, 5))
sns.histplot(df_precio['VALOR'], bins=50, kde=True, color='steelblue')
plt.title('Distribución del índice de precio mediano de vivienda en España')
plt.xlabel('Índice de precio')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('src/img/01_distribucion_precios.png', dpi=150, bbox_inches='tight')
plt.show()

### 📌 Conclusiones — Distribución general

- La mayoría de municipios se concentran entre índice **250 y 500**
- La distribución presenta **asimetría positiva** (cola derecha larga)
- Existe un grupo reducido de municipios con índices superiores a 1.000, correspondientes a zonas de alta tensión inmobiliaria
- Esta estructura **dual** del mercado (mayoría moderada + minoría extrema) distorsiona la media nacional al alza


## 4. Análisis bivariante — Hipótesis

### H3 — El precio del alquiler ha aumentado progresivamente con el tiempo

**Hipótesis:** El índice de precio mediano nacional aumenta año a año entre 2011 y 2024.


In [ ]:
# Evolución temporal: mediana del índice por año
evolucion = df_precio.groupby('AÑO')['VALOR'].median()

plt.figure(figsize=(10, 5))
sns.lineplot(x=evolucion.index, y=evolucion.values, marker='o', color='steelblue', linewidth=2)
plt.title('Evolución del índice de precio mediano en España (2011–2024)')
plt.xlabel('Año')
plt.ylabel('Índice de precio mediano')
plt.xticks(evolucion.index, rotation=45)
plt.tight_layout()
plt.savefig('src/img/02_evolucion_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

print("Valores por año:")
print(evolucion.round(1))

### ✅ H3 Confirmada

- **2011–2015:** caída del índice — consecuencia directa de la crisis inmobiliaria post-2008
- **2015–2019:** recuperación gradual sostenida por el crecimiento económico
- **2020–2024:** aceleración intensa — el índice de 2024 (~425) es el **más alto de toda la serie histórica**
- La subida no es lineal, pero la tendencia general es alcista y el máximo se alcanza en 2024


### H1 — Las grandes provincias tienen precios de alquiler superiores

**Hipótesis:** Madrid y Barcelona, por ser las provincias más grandes y pobladas, lideran el ranking de precios.


In [ ]:
# Ranking de las 15 provincias con mayor índice mediano
top_provincias = df_precio.groupby('PROVINCIA')['VALOR'].median().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_provincias.values, y=top_provincias.index, palette='Blues_r')
plt.title('Top 15 provincias con mayor índice de precio mediano')
plt.xlabel('Índice de precio mediano')
plt.ylabel('Provincia')
plt.tight_layout()
plt.savefig('src/img/03_ranking_provincias.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 provincias más caras:")
print(top_provincias.head(5).round(1))

### ⚠️ H1 Parcialmente confirmada

- **Bizkaia (~640) y Gipuzkoa (~600)** lideran el ranking por encima de Madrid y Barcelona (~510)
- Las grandes capitales sí están en el top, pero **no son las más caras**
- El País Vasco, con su alto nivel de renta y calidad de vida, presenta el mercado más tensionado de España
- Este es el **hallazgo más inesperado** del análisis y cuestiona la narrativa habitual sobre el problema de la vivienda


### H2 — Existe una desigualdad territorial clara entre provincias

**Hipótesis:** La diferencia de precios entre provincias es muy pronunciada y estadísticamente significativa.


In [ ]:
# Boxplot por provincia ordenado por mediana
precio_provincia = df_precio.groupby('PROVINCIA')['VALOR'].median().sort_values(ascending=False)

plt.figure(figsize=(14, 10))
sns.boxplot(data=df_precio, x='VALOR', y='PROVINCIA', order=precio_provincia.index, 
            palette='Blues_r', fliersize=2)
plt.title('Distribución del índice de precio por provincia')
plt.xlabel('Índice de precio mediano')
plt.ylabel('Provincia')
plt.tight_layout()
plt.savefig('src/img/04_desigualdad_territorial.png', dpi=150, bbox_inches='tight')
plt.show()

# Cuantificar la brecha
max_prov = precio_provincia.index[0]
min_prov = precio_provincia.index[-1]
print(f"Provincia más cara: {max_prov} ({precio_provincia.iloc[0]:.0f})")
print(f"Provincia más barata: {min_prov} ({precio_provincia.iloc[-1]:.0f})")
print(f"Brecha: {precio_provincia.iloc[0] / precio_provincia.iloc[-1]:.1f}x")

### ✅ H2 Confirmada — Hallazgo más robusto del análisis

- La brecha entre Bizkaia (~640) y Lugo (~200) supera el **triple**
- Provincias del interior (Teruel, Ávila, Badajoz, Lugo) muestran cajas compactas y medianas bajas
- Madrid y Barcelona destacan no solo por precios altos sino por **amplísima dispersión interna**
- La desigualdad es **doble**: entre provincias y dentro de cada provincia


### H4 — El tipo de vivienda influye en el precio del alquiler

**Hipótesis:** La vivienda unifamiliar presenta un índice de precio significativamente superior a la colectiva.


In [ ]:
# Comparativa por tipo de vivienda
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_precio, x='TIPO_VIVIENDA', y='VALOR', palette='Set2')
plt.title('Índice de precio según tipo de vivienda')
plt.xlabel('Tipo de vivienda')
plt.ylabel('Índice de precio mediano')
plt.tight_layout()
plt.savefig('src/img/05_tipo_vivienda.png', dpi=150, bbox_inches='tight')
plt.show()

# Estadísticas por tipo
print(df_precio.groupby('TIPO_VIVIENDA')['VALOR'].describe().round(1))

### ⚠️ H4 Confirmada con matiz

- Las **medianas son similares** (~360 colectiva vs ~380 unifamiliar): en el segmento típico la diferencia es moderada
- La **variabilidad** de la unifamiliar es significativamente mayor — outliers que alcanzan 2.200+
- El tipo de vivienda influye principalmente en los **extremos del mercado** (segmento premium)
- La unifamiliar es el activo más volátil: sube más en auge y baja más en crisis


## 5. Análisis multivariante

In [ ]:
# Evolución del precio por año y tipo de vivienda
evolucion_tipo = df_precio.groupby(['AÑO', 'TIPO_VIVIENDA'])['VALOR'].median().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=evolucion_tipo, x='AÑO', y='VALOR', hue='TIPO_VIVIENDA', 
             marker='o', linewidth=2)
plt.title('Evolución del índice de precio por tipo de vivienda (2011–2024)')
plt.xlabel('Año')
plt.ylabel('Índice de precio mediano')
plt.xticks(evolucion_tipo['AÑO'].unique(), rotation=45)
plt.tight_layout()
plt.savefig('src/img/06_evolucion_tipo_vivienda.png', dpi=150, bbox_inches='tight')
plt.show()

### 📌 Conclusiones — Evolución por tipo de vivienda

- La **unifamiliar** mantiene precios superiores a la colectiva en todo el período analizado
- Ambos tipos siguen la misma tendencia general: caída hasta 2015 y recuperación sostenida posterior
- A partir de 2021 ambos se disparan, pero la **colectiva acelera más proporcionalmente**, reduciendo ligeramente la brecha


In [ ]:
# Heatmap de correlación entre variables numéricas
plt.figure(figsize=(7, 5))
corr = df_precio[['AÑO', 'VALOR']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', 
            square=True, linewidths=0.5)
plt.title('Heatmap de correlación — AÑO vs VALOR')
plt.tight_layout()
plt.savefig('src/img/07_heatmap_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Correlación de Pearson AÑO–VALOR: {df_precio['AÑO'].corr(df_precio['VALOR']):.2f}")

In [ ]:
# Scatterplot: relación entre año y precio por tipo de vivienda
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_precio, x='AÑO', y='VALOR', 
                hue='TIPO_VIVIENDA', alpha=0.3, s=10)
plt.title('Relación entre año y precio según tipo de vivienda')
plt.xlabel('Año')
plt.ylabel('Índice de precio mediano')
plt.tight_layout()
plt.savefig('src/img/08_scatterplot_anyo_precio.png', dpi=150, bbox_inches='tight')
plt.show()

### 📌 Conclusiones — Correlación y dispersión

- La correlación AÑO–VALOR es **r = 0,19**: positiva pero débil
- La baja correlación se explica por la caída 2011–2015 que rompe la linealidad
- La **ubicación geográfica** (provincia) tiene un peso explicativo muy superior al paso del tiempo
- Implicación directa: las políticas de vivienda deben ser **territorialmente diferenciadas**


## 6. Validación de hipótesis

| Hipótesis | Enunciado | Resultado | Evidencia |
|-----------|-----------|-----------|-----------|
| **H1** | Las grandes provincias tienen precios superiores | ⚠️ Parcialmente confirmada | Bizkaia (~640) y Gipuzkoa (~600) superan a Madrid y Barcelona (~510) |
| **H2** | Existe desigualdad territorial clara | ✅ Confirmada | Brecha Bizkaia–Lugo > 3x. Hallazgo más robusto del análisis |
| **H3** | El precio aumenta progresivamente | ✅ Confirmada | Índice 2024 (~425) es el máximo histórico de la serie |
| **H4** | El tipo de vivienda influye en el precio | ⚠️ Confirmada con matiz | Medianas similares, pero variabilidad y techo muy superiores en unifamiliar |


## 7. Conclusiones finales

### 1. La ubicación geográfica es el factor dominante
La provincia explica una fracción de la varianza en el precio muy superior a cualquier otra variable. Un piso en Bizkaia puede costar más del triple que uno equivalente en Lugo. Esta desigualdad se ha intensificado en la última década.

### 2. El alquiler en España vive su momento más caro de la historia reciente
El índice de 2024 supera todos los valores anteriores, incluyendo los previos a 2008. La aceleración 2020–2024 se produce en un contexto de inflación elevada y contracción del poder adquisitivo.

### 3. El problema del alquiler no es exclusivo de Madrid y Barcelona
El País Vasco lidera el ranking con Bizkaia y Gipuzkoa por encima de las dos grandes metrópolis. El debate público debería ampliar su foco geográfico.

### 4. La vivienda unifamiliar es el segmento más volátil
La diferencia colectiva/unifamiliar es moderada en el segmento típico, pero muy pronunciada en el premium. La unifamiliar sube más, baja más en crisis y alcanza valores extremos más altos.

---

### 🔎 Limitaciones del estudio
- El índice utilizado es una referencia oficial, no el precio real transaccionado
- No se incorporan variables socioeconómicas complementarias (renta, demografía)
- El análisis es exclusivamente descriptivo — no se desarrollan modelos predictivos
- El enfoque provincial puede ocultar diferencias intermunicipales relevantes
